# 01 — Exploration et qualification des sources de données

**Auteur :** Benoit Girard — Ingénieur Data junior, CheckItAI  
**Projet OpenClassrooms n°12 :** *Extrayez des données multimodales de sites web*

Ce notebook accompagne le **livrable n°1**. Il vérifie concrètement que les sources retenues fournissent bien des données **multimodales** (texte + image) exploitables pour entraîner un détecteur de fake news. Le rapport détaillé se trouve dans `docs/rapport_exploration_sources.md`.

## Définition du *done*

| Critère | Cible |
|---|---|
| Au moins 3 sources multimodales qualifiées | ✅ |
| Présence vérifiée de texte **et** d'image par publication | ✅ |
| Au moins une source labellisée (vérité terrain) | ✅ |
| Méthodes d'extraction identifiées | ✅ |

In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))

from dotenv import load_dotenv

load_dotenv(ROOT / ".env")

from checkitai.logging_setup import setup_logging

setup_logging()

## 1. Flux RSS de presse — source officielle, multimodale, sans clé

Les flux RSS exposent, pour chaque article, un **titre**, un **résumé** et souvent une **image** (`media:content`, `enclosure` ou `<img>` dans le résumé). On lit un flux avec `feedparser` et on extrait l'image avec une fonction dédiée.

In [2]:
from checkitai.config import ExtractionConfig
from checkitai.sources.rss import fetch_rss_feed

config = ExtractionConfig()
echantillon = fetch_rss_feed("bbc_news", dict(config.rss_feeds)["bbc_news"], config)
print(f"{len(echantillon)} publications lues depuis BBC News")
exemple = next(p for p in echantillon if p["image_url"])
for cle in ("title", "image_url", "url"):
    print(f"{cle:10s}: {str(exemple[cle])[:90]}")

2026-06-29 18:06:42 | INFO    | checkitai.sources.rss | RSS : lecture du flux 'bbc_news' (https://feeds.bbci.co.uk/news/world/rss.xml)


2026-06-29 18:06:42 | INFO    | checkitai.sources.rss | RSS : 38 publications recuperees depuis 'bbc_news'


38 publications lues depuis BBC News
title     : Mum rescued from Venezuela rubble with newborn baby tells BBC how he helped her survive
image_url : https://ichef.bbci.co.uk/ace/standard/240/cpsprodpb/bcfd/live/a0f94100-739f-11f1-8546-8f19
url       : https://www.bbc.co.uk/news/articles/clyw3rkj2p7o?at_medium=RSS&at_campaign=rss


On constate qu'une entrée RSS porte bien **du texte et une image** : la modalité visuelle est donc disponible dès la source.

## 2. API NewsData.io — actualité multimodale via REST/JSON

NewsData.io renvoie des articles au format JSON avec un champ `image_url` direct. La source ne s'active que si une clé `NEWSDATA_API_KEY` est présente.

In [3]:
from checkitai.sources import newsdata

print("Source NewsData.io activée :", newsdata.is_enabled())
articles = newsdata.fetch_newsdata(config)
print(f"{len(articles)} articles récupérés")
if articles:
    a = articles[0]
    print("Titre :", a["title"][:90])
    print("Image :", a["image_url"][:90])

Source NewsData.io activée : True
2026-06-29 18:06:42 | INFO    | checkitai.sources.newsdata | NewsData.io : appel de l'API (https://newsdata.io/api/1/news)


2026-06-29 18:06:43 | INFO    | checkitai.sources.newsdata | NewsData.io : 10 articles recuperes


10 articles récupérés
Titre : Young Eagle's season over amid concussion concerns
Image : https://resources.afl.com.au/photo-resources/2026/03/22/32642fb4-0aca-48ee-b53f-f835240b53


## 3. FakeNewsNet — la vérité terrain (real / fake)

FakeNewsNet est la **seule source labellisée**. C'est elle qui rend possible l'apprentissage supervisé. On charge l'échantillon versionné.

In [4]:
import pandas as pd

from checkitai.sources.fakenewsnet import fetch_fakenewsnet

labellisees = fetch_fakenewsnet(config)
df = pd.DataFrame(labellisees)
print(f"{len(df)} publications labellisées")
df["label"].value_counts()

2026-06-29 18:06:46 | INFO    | checkitai.sources.fakenewsnet | FakeNewsNet : utilisation de l'echantillon versionne fakenewsnet_sample.csv


2026-06-29 18:06:46 | INFO    | checkitai.sources.fakenewsnet | FakeNewsNet : 24 publications labellisees chargees


24 publications labellisées


label
fake    12
real    12
Name: count, dtype: int64

## Conclusion

Les trois sources intégrées sont complémentaires : **RSS** et **NewsData.io** apportent un flux frais et multimodal ; **FakeNewsNet** apporte les labels. Toutes reposent sur des **canaux officiels** (pas de scraping). La stratégie complète est argumentée dans le livrable n°1.